# Mission Control — Laboratório eBPF (libbpf) + Docker

Requisitos no **anfitrião Linux**: `clang`, `llvm`, `bpftool`, `libbpf` (dev), `docker` / `docker compose`, `sudo` para eBPF/TC, kernel com BTF (`/sys/kernel/btf/vmlinux`).

Execute as células em ordem com o diretório de trabalho na pasta **`mission-control-lab`** (onde estão `Makefile`, `docker-compose.yml` e este notebook).

O `make` gera `vmlinux.h`, o bytecode TC, `deadlock_bpf.skel.h` e o binário `loader`.

In [ ]:
!make clean 2>/dev/null; make all

### 2. Subir os contentores (rede `mission_control_net`)

In [ ]:
!docker compose up -d --build

### 3. Estado dos contentores e rede

In [ ]:
!docker ps -a --filter "name=mc_app" && docker network inspect mission_control_net --format '{{json .}}' | head -c 2000

### 4. Loader eBPF (TC na bridge Docker + ringbuf)

O programa anexa-se à interface `br-<id>` da rede. Pode passar o nome explicitamente ou deixar o loader detetar `br-*` na sub-rede 172.28.0.0/16.

**Nota:** esta célula bloqueia até Ctrl+C ou interrupção do kernel; noutro terminal pode ver `docker logs`.

In [ ]:
# Bridge Docker: br- + primeiros 12 caracteres do ID da rede
!BR=$(docker network inspect mission_control_net -f '{{.Id}}' | head -c 12) && echo "Bridge: br-$BR" && sudo ./loader "br-$BR"

### 5. Logs dos contentores (RST / fim do `recv`)

Depois de ~5 s com ligações em `ESTABLISHED` sem carga TCP, o kernel deve emitir o alerta e reescrever um segmento para RST; os processos saem de `recv`.

In [ ]:
!docker logs mc_app_a 2>&1 | tail -n 50
!echo '---'
!docker logs mc_app_b 2>&1 | tail -n 50

### Término opcional

In [ ]:
!docker compose down